INSTALL THE NECESSARY LIBRARAIES


In [1]:
!pip install graphrag datasets openai pandas numpy -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install Pillow==10.4.0 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 10.4.0 which is incompatible.


In [3]:
!pip install transformers torch -q

In [4]:
!pip install openai -q

In [5]:
!pip install graphrag -q

In [6]:
!pip install sentence-transformers faiss-cpu -q

In [22]:
pip install --upgrade --force-reinstall pyarrow datasets

  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [7]:
!pip install Pillow==10.4.0 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 10.4.0 which is incompatible.


Import the necessary dataset

In [8]:
from datasets import load_dataset

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

print(dataset)
print("\nExample question:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})

Example question:
{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?', 'answer': 'Nitrofurantoin', 'options': {'A': 'Ampicillin', 'B': 

BASELINE LLM TEST(NO RAG)

In [9]:
import pandas as pd
import time
from openai import OpenAI

client_groq = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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"
)

def ask_baseline_llm(question, options, retries=3):
    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. Answer this clinical question.

Question: {question}

Options:
{options_text}

Reply with only the letter A, B, C, or D. Nothing else."""

    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5,
                temperature=0
            )
            text = response.choices[0].message.content.strip()
            for char in text:
                if char in ['A', 'B', 'C', 'D']:
                    return char
            return "X"
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(10)
    return "X"

# Test on 20 examples
test_sample = dataset['test'].select(range(20))

results = []
for i, item in enumerate(test_sample):
    predicted = ask_baseline_llm(item['question'], item['options'])
    correct = item['answer_idx']
    results.append({
        'predicted': predicted,
        'correct': correct,
        'is_correct': predicted == correct
    })
    print(f"{i+1}/20 - Predicted: {predicted} Correct: {correct}")
    time.sleep(2)

df_baseline = pd.DataFrame(results)
print(f"\nBaseline LLM Accuracy (20 samples): {df_baseline['is_correct'].mean()*100:.1f}%")

1/20 - Predicted: A Correct: B
2/20 - Predicted: D Correct: D
3/20 - Predicted: B Correct: B
4/20 - Predicted: B Correct: D
5/20 - Predicted: B Correct: B
6/20 - Predicted: B Correct: D
7/20 - Predicted: B Correct: C
8/20 - Predicted: C Correct: C
9/20 - Predicted: B Correct: B
10/20 - Predicted: A Correct: A
11/20 - Predicted: D Correct: D
12/20 - Predicted: D Correct: D
13/20 - Predicted: B Correct: B
14/20 - Predicted: D Correct: D
15/20 - Predicted: C Correct: C
16/20 - Predicted: B Correct: B
17/20 - Predicted: D Correct: D
18/20 - Predicted: D Correct: D
19/20 - Predicted: A Correct: B
20/20 - Predicted: A Correct: D

Baseline LLM Accuracy (20 samples): 70.0%


RAG setup

In [10]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Use MedQA training set as knowledge base
knowledge_base = dataset['train'].select(range(500))

# Create documents from training data
documents = []
for item in knowledge_base:
    doc = f"Question: {item['question']}\nCorrect Answer: {item['answer']}"
    documents.append(doc)

print(f"Knowledge base: {len(documents)} documents")

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
print("Creating embeddings...")
embeddings = embedder.encode(documents, show_progress_bar=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))
print(f"RAG index ready with {index.ntotal} vectors!")

Knowledge base: 500 documents


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

RAG index ready with 500 vectors!


RAG implementataion and testing

In [11]:
def ask_rag_llm(question, options, k=3):
    # Step 1: Retrieve relevant documents
    query_embedding = embedder.encode([question])
    distances, indices = index.search(query_embedding.astype('float32'), k)
    retrieved_docs = [documents[i] for i in indices[0]]
    context = "\n\n".join(retrieved_docs)

    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. Use the following reference cases to answer the question.

Reference Cases:
{context}

Question: {question}

{options_text}

Reply with only the letter A, B, C, or D. Nothing else."""

    for attempt in range(3):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5,
                temperature=0
            )
            text = response.choices[0].message.content.strip()
            for char in text:
                if char in ['A', 'B', 'C', 'D']:
                    return char
            return "X"
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(10)
    return "X"

# Test RAG on same 20 examples
test_sample = dataset['test'].select(range(20))

rag_results = []
for i, item in enumerate(test_sample):
    predicted = ask_rag_llm(item['question'], item['options'])
    correct = item['answer_idx']
    rag_results.append({
        'predicted': predicted,
        'correct': correct,
        'is_correct': predicted == correct
    })
    print(f"{i+1}/20 - Predicted: {predicted} Correct: {correct}")
    time.sleep(2)

df_rag = pd.DataFrame(rag_results)
print(f"\nBaseline LLM Accuracy: 70.0%")
print(f"RAG Accuracy (20 samples): {df_rag['is_correct'].mean()*100:.1f}%")

1/20 - Predicted: A Correct: B
2/20 - Predicted: D Correct: D
3/20 - Predicted: B Correct: B
4/20 - Predicted: B Correct: D
5/20 - Predicted: B Correct: B
6/20 - Predicted: B Correct: D
7/20 - Predicted: B Correct: C
8/20 - Predicted: C Correct: C
9/20 - Predicted: B Correct: B
10/20 - Predicted: B Correct: A
11/20 - Predicted: D Correct: D
Attempt 1 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kv6yhhy8fkbb6jgas1rkvyb4` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99050, Requested 1242. Please try again in 4m12.287999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Attempt 2 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kv6yhhy8fkbb6jgas1rkvyb4` service tier `on_demand` on tokens per day (TPD): Limit 10000

GRAPH RAG SETUP

In [12]:
!pip install botocore

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 110.9 MB/s eta 0:00:00


In [13]:
!apt-get install -y zstd -q

# Now install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,289 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama 

In [14]:
import os

settings_content = """
completion_models:
  default_completion_model:
    model_provider: openai
    model: llama-3.3-70b-versatile
    auth_method: api_key
    api_key: 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
    api_base: https://api.groq.com/openai/v1
    requests_per_minute: 20
    tokens_per_minute: 5000
    max_retries: 3
    retry:
      type: exponential_backoff

embedding_models:
  default_embedding_model:
    model_provider: openai
    model: nomic-embed-text
    auth_method: api_key
    api_key: ollama
    api_base: http://localhost:11434/v1
    retry:
      type: exponential_backoff
"""

os.makedirs("/content/graphrag_project", exist_ok=True)   # <-- the fix

with open("/content/graphrag_project/settings.yaml", "w") as f:
    f.write(settings_content)

print("Settings updated!")

Settings updated!


SETUP OLLAMA SERVER

In [15]:
import subprocess
import time

# Kill any existing ollama process
!pkill ollama
time.sleep(2)

# Restart ollama - it should now detect the GPU
ollama_process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

print("Ollama restarted!")

# Pull models
!ollama pull nomic-embed-text
!ollama pull llama3.2:3b

print("Models ready!")

Ollama restarted!


Models ready!


TEST SERVER

In [16]:
# Run a quick test and check GPU usage
import subprocess

# Check ollama ps to see if model is loaded on GPU
result = subprocess.run(['ollama', 'ps'], capture_output=True, text=True)
print(result.stdout)

# Quick test query to load model into GPU memory
import requests
response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3.2:3b", "prompt": "test", "stream": False}
)
print("\nTest response received:", response.status_code)

# Check ollama ps again
result2 = subprocess.run(['ollama', 'ps'], capture_output=True, text=True)
print(result2.stdout)

NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


Test response received: 200
NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.2:3b    a80c4f17acd5    2.6 GB    100% GPU     4096       4 minutes from now    



GraphRAG indexing and implementation

In [17]:
!pip uninstall openai graphrag litellm -y -q
!pip install graphrag==0.3.6 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/91

In [20]:
import networkx as nx
from openai import OpenAI
import json
import time
import pandas as pd
from datasets import load_dataset
import re

client_groq = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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"
)

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

def extract_entities_and_relationships(text, retries=3):
    prompt = f"""Extract medical entities and relationships from this clinical text.
Return ONLY a JSON object like this:
{{
  "entities": [
    {{"name": "entity name", "type": "disease|symptom|medication|treatment|procedure"}}
  ],
  "relationships": [
    {{"source": "entity1", "target": "entity2", "relation": "treats|causes|indicates|contraindicates"}}
  ]
}}

Text: {text[:500]}

JSON only, no explanation:"""
    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=500,
                temperature=0
            )
            text_response = response.choices[0].message.content.strip()
            text_response = text_response.replace("```json", "").replace("```", "").strip()
            match = re.search(r'\{.*\}', text_response, re.DOTALL)
            if match:
                text_response = match.group()
            return json.loads(text_response)
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(5)
    return {"entities": [], "relationships": []}

print("Building knowledge graph...")
G = nx.Graph()
knowledge_base = dataset['train'].select(range(50))

for i, item in enumerate(knowledge_base):
    text = f"{item['question']} Answer: {item['answer']}"
    extracted = extract_entities_and_relationships(text)
    for ent in extracted.get("entities", []):
        G.add_node(ent["name"], type=ent["type"])
    for rel in extracted.get("relationships", []):
        if rel["source"] in G and rel["target"] in G:
            G.add_edge(rel["source"], rel["target"], relation=rel["relation"])
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/50 docs | Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

print(f"\nGraph built! Entities: {G.number_of_nodes()}, Relationships: {G.number_of_edges()}")

Building knowledge graph...
Processed 10/50 docs | Nodes: 49 | Edges: 35
Processed 20/50 docs | Nodes: 116 | Edges: 89
Processed 30/50 docs | Nodes: 171 | Edges: 123
Processed 40/50 docs | Nodes: 232 | Edges: 174
Processed 50/50 docs | Nodes: 275 | Edges: 209

Graph built! Entities: 275, Relationships: 209


GRAPH RAG EVALUATION

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import time

# Load embedder and pre-embed all graph nodes
embedder = SentenceTransformer('all-MiniLM-L6-v2')
node_list = list(G.nodes())
node_embeddings = embedder.encode(node_list)
print(f"Embedded {len(node_list)} graph nodes!")

# Semantic graph context retrieval
def get_graph_context_semantic(question, top_k=5):
    question_embedding = embedder.encode([question])

    # Cosine similarity
    sims = np.dot(node_embeddings, question_embedding.T).flatten()
    top_indices = sims.argsort()[-top_k:][::-1]
    relevant_nodes = [node_list[i] for i in top_indices]

    context_parts = []
    for node in relevant_nodes:
        neighbors = list(G.neighbors(node))
        node_type = G.nodes[node].get('type', 'unknown')
        if neighbors:
            edges_info = []
            for neighbor in neighbors[:3]:
                relation = G.edges[node, neighbor].get('relation', 'related_to')
                edges_info.append(f"{neighbor} ({relation})")
            context_parts.append(f"{node} [{node_type}] → {', '.join(edges_info)}")
        else:
            context_parts.append(f"{node} [{node_type}]")

    return "\n".join(context_parts)

# GraphRAG query function
def ask_graphrag_custom(question, options, retries=3):
    graph_context = get_graph_context_semantic(question)
    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. Use the knowledge graph context below to answer the question.

Knowledge Graph Context:
{graph_context}

Question: {question}

Options:
{options_text}

Reply with only the letter A, B, C, or D. Nothing else."""

    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5,
                temperature=0
            )
            text = response.choices[0].message.content.strip()
            for char in text:
                if char in ['A', 'B', 'C', 'D']:
                    return char
            return "X"
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(10)
    return "X"

# Evaluation on 20 test samples
test_sample = dataset['test'].select(range(20))

graphrag_results = []
for i, item in enumerate(test_sample):
    predicted = ask_graphrag_custom(item['question'], item['options'])
    correct = item['answer_idx']
    graphrag_results.append({
        'predicted': predicted,
        'correct': correct,
        'is_correct': predicted == correct
    })
    print(f"{i+1}/20 - Predicted: {predicted} | Correct: {correct} | {'✓' if predicted == correct else '✗'}")
    time.sleep(2)

df_graphrag = pd.DataFrame(graphrag_results)

# Final comparison
print("\n=== FINAL COMPARISON ===")
print(f"Baseline LLM : 70.0%")
print(f"RAG          : 65.0%")
print(f"GraphRAG     : {df_graphrag['is_correct'].mean()*100:.1f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedded 275 graph nodes!
1/20 - Predicted: A | Correct: B | ✗
2/20 - Predicted: D | Correct: D | ✓
3/20 - Predicted: B | Correct: B | ✓
4/20 - Predicted: B | Correct: D | ✗
5/20 - Predicted: B | Correct: B | ✓
6/20 - Predicted: B | Correct: D | ✗
7/20 - Predicted: B | Correct: C | ✗
8/20 - Predicted: C | Correct: C | ✓
9/20 - Predicted: B | Correct: B | ✓
10/20 - Predicted: A | Correct: A | ✓
11/20 - Predicted: D | Correct: D | ✓
12/20 - Predicted: D | Correct: D | ✓
13/20 - Predicted: B | Correct: B | ✓
14/20 - Predicted: D | Correct: D | ✓
15/20 - Predicted: C | Correct: C | ✓
16/20 - Predicted: B | Correct: B | ✓
17/20 - Predicted: D | Correct: D | ✓
18/20 - Predicted: D | Correct: D | ✓
19/20 - Predicted: A | Correct: B | ✗
20/20 - Predicted: B | Correct: D | ✗

=== FINAL COMPARISON ===
Baseline LLM : 70.0%
RAG          : 65.0%
GraphRAG     : 70.0%


GRAPH RAG USAGE EXAMPLES

In [22]:
# Test query on our custom GraphRAG
test_question = "What medications are used for high blood pressure?"

context = get_graph_context_semantic(test_question)
print("=== Graph Context Retrieved ===")
print(context)

response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a medical expert."},
        {"role": "user", "content": f"""Use this knowledge graph context to answer the question.

Knowledge Graph Context:
{context}

Question: {test_question}

Provide a detailed answer."""}
    ],
    max_tokens=300,
    temperature=0
)
print("\n=== GraphRAG Answer ===")
print(response.choices[0].message.content)

=== Graph Context Retrieved ===
hypertension [disease] → oral contraceptive pills (contraindicates), enlarged thyroid gland (causes), ramipril (treats)
low blood pressure [symptom] → dizziness (causes), increased pulse rate (indicates), increased respiration rate (indicates)
arterial hypertension [disease] → myocardial infarction (causes), atorvastatin (treats), enalapril (treats)
metoprolol [medication] → chronic heart failure (treats)
risperidone [medication] → abnormal behavior (treats), delusions (treats), aggression (treats)

=== GraphRAG Answer ===
Based on the provided knowledge graph context, several medications are used to treat high blood pressure, also known as hypertension or arterial hypertension. These medications include:

1. **Ramipril**: This medication is specifically mentioned as a treatment for hypertension. Ramipril belongs to a class of drugs known as angiotensin-converting enzyme (ACE) inhibitors, which work by relaxing blood vessels and reducing blood pressure.


SELF CORRECTION GROUP

In [23]:
def get_answer_confidence(question, options, predicted_answer, graph_context):
    """Ask LLM to rate its own confidence in the answer"""
    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. You previously answered this question as {predicted_answer}.

Knowledge Graph Context Used:
{graph_context}

Question: {question}

Options:
{options_text}

Rate your confidence in answer {predicted_answer} on a scale of 1-10.
Reply with ONLY a single integer between 1 and 10. Nothing else."""

    try:
        response = client_groq.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5,
            temperature=0
        )
        text = response.choices[0].message.content.strip()
        for char in text:
            if char.isdigit():
                return int(char)
        return 5
    except:
        return 5

def ask_agentic_graphrag(question, options, retries=3, confidence_threshold=6):
    # Step 1: Initial GraphRAG answer
    graph_context = get_graph_context_semantic(question)
    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. Use the knowledge graph context below to answer the question.

Knowledge Graph Context:
{graph_context}

Question: {question}

Options:
{options_text}

Reply with only the letter A, B, C, or D. Nothing else."""

    predicted = "X"
    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5,
                temperature=0
            )
            text = response.choices[0].message.content.strip()
            for char in text:
                if char in ['A', 'B', 'C', 'D']:
                    predicted = char
                    break
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(10)

    if predicted == "X":
        return "X", 0   # <-- return a tuple even on failure, so unpacking never breaks

    # Step 2: Check confidence
    time.sleep(2)
    confidence = get_answer_confidence(question, options, predicted, graph_context)
    initial_confidence = confidence   # store before any correction overwrites context

    # Step 3: Self-correct if low confidence
    if confidence < confidence_threshold:
        print(f"    Low confidence ({confidence}/10) - triggering self-correction...")

        expanded_context = get_graph_context_semantic(question, top_k=10)

        correction_prompt = f"""You are a medical expert reviewing your previous answer.

Your previous answer was {predicted} but you had low confidence.

Expanded Knowledge Graph Context:
{expanded_context}

Question: {question}

Options:
{options_text}

Review carefully and provide your best answer.
Reply with only the letter A, B, C, or D. Nothing else."""

        for attempt in range(retries):
            try:
                response = client_groq.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                        {"role": "user", "content": correction_prompt}
                    ],
                    max_tokens=5,
                    temperature=0
                )
                text = response.choices[0].message.content.strip()
                for char in text:
                    if char in ['A', 'B', 'C', 'D']:
                        predicted = char
                        break
                break
            except Exception as e:
                print(f"Correction attempt {attempt+1} failed: {e}")
                time.sleep(10)

    return predicted, initial_confidence   # <-- always returns a tuple now

# Evaluate on same 20 test samples
agentic_results = []
test_sample = dataset['test'].select(range(20))

for i, item in enumerate(test_sample):
    predicted, confidence = ask_agentic_graphrag(item['question'], item['options'])
    correct = item['answer_idx']
    agentic_results.append({
        'predicted': predicted,
        'correct': correct,
        'is_correct': predicted == correct,
        'confidence': confidence,
    })
    print(f"{i+1}/20 - Predicted: {predicted} | Correct: {correct} | Confidence: {confidence}")
    time.sleep(3)

df_agentic = pd.DataFrame(agentic_results)

    Low confidence (1/10) - triggering self-correction...
1/20 - Predicted: A | Correct: B | Confidence: 1
2/20 - Predicted: D | Correct: D | Confidence: 8
3/20 - Predicted: B | Correct: B | Confidence: 9
    Low confidence (1/10) - triggering self-correction...
4/20 - Predicted: B | Correct: D | Confidence: 1
5/20 - Predicted: B | Correct: B | Confidence: 9
6/20 - Predicted: B | Correct: D | Confidence: 8
7/20 - Predicted: B | Correct: C | Confidence: 8
    Low confidence (1/10) - triggering self-correction...
8/20 - Predicted: C | Correct: C | Confidence: 1
    Low confidence (1/10) - triggering self-correction...
9/20 - Predicted: B | Correct: B | Confidence: 1
10/20 - Predicted: A | Correct: A | Confidence: 8
    Low confidence (1/10) - triggering self-correction...
11/20 - Predicted: D | Correct: D | Confidence: 1
    Low confidence (1/10) - triggering self-correction...
12/20 - Predicted: D | Correct: D | Confidence: 1
    Low confidence (1/10) - triggering self-correction...
13/

In [24]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import time
import pandas as pd


def fetch_pubmed_abstracts(query, max_results=3):
    """Fetch abstracts from PubMed using NCBI Entrez API (no key needed)"""
    try:
        search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={urllib.parse.quote(query)}&retmax={max_results}&retmode=xml"
        with urllib.request.urlopen(search_url, timeout=10) as response:
            search_xml = response.read()

        root = ET.fromstring(search_xml)
        pmids = [id_elem.text for id_elem in root.findall('.//Id')]

        if not pmids:
            return []

        ids_str = ",".join(pmids)
        fetch_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id={ids_str}&rettype=abstract&retmode=xml"
        with urllib.request.urlopen(fetch_url, timeout=10) as response:
            fetch_xml = response.read()

        root = ET.fromstring(fetch_xml)
        abstracts = []
        for article in root.findall('.//PubmedArticle'):
            abstract_text = ""
            for text in article.findall('.//AbstractText'):
                if text.text:
                    abstract_text += text.text + " "
            if abstract_text.strip():
                abstracts.append(abstract_text.strip()[:500])

        return abstracts
    except Exception as e:
        print(f"PubMed fetch failed: {e}")
        return []


def extract_pubmed_keywords(question, retries=2):
    """Extract 2-4 clinically searchable keywords from a vignette (MeSH-style, not prose)."""
    prompt = f"""Extract 2 to 4 key medical search terms from this clinical vignette,
suitable for a PubMed search (e.g., disease names, drug names, key symptoms).
Return ONLY a comma-separated list, nothing else.

Vignette: {question[:600]}

Keywords:"""
    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=40,
                temperature=0
            )
            keywords = response.choices[0].message.content.strip()
            keywords = keywords.replace("\n", " ").strip()
            if keywords:
                return keywords
        except Exception as e:
            print(f"Keyword extraction attempt {attempt+1} failed: {e}")
            time.sleep(5)
    return question[:100]  # fallback to old behavior if extraction fails


def update_graph_from_pubmed(question, G, node_list, node_embeddings):
    """Fetch PubMed abstracts and dynamically add entities to graph"""
    search_query = extract_pubmed_keywords(question)
    print(f"    PubMed query: {search_query}")
    abstracts = fetch_pubmed_abstracts(search_query)
    print(f"    Abstracts fetched: {len(abstracts)}")

    new_nodes = 0
    new_edges = 0

    for abstract in abstracts:
        extracted = extract_entities_and_relationships(abstract)
        for ent in extracted.get("entities", []):
            if ent["name"] not in G:
                G.add_node(ent["name"], type=ent["type"])
                node_list.append(ent["name"])
                new_nodes += 1
        for rel in extracted.get("relationships", []):
            if rel["source"] in G and rel["target"] in G:
                G.add_edge(rel["source"], rel["target"], relation=rel["relation"])
                new_edges += 1

    if new_nodes > 0:
        new_embeddings = embedder.encode(node_list)
        return new_embeddings, new_nodes, new_edges

    return node_embeddings, new_nodes, new_edges


def ask_agentic_graphrag_pubmed(question, options, retries=3, confidence_threshold=6):
    global node_list, node_embeddings

    # Step 1: Dynamic graph update from PubMed
    print(f"    Fetching PubMed abstracts...")
    node_embeddings, new_nodes, new_edges = update_graph_from_pubmed(
        question, G, node_list, node_embeddings
    )
    if new_nodes > 0:
        print(f"    Graph updated: +{new_nodes} entities, +{new_edges} relationships")

    # Step 2: Initial GraphRAG answer with updated graph
    graph_context = get_graph_context_semantic(question)
    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. Use the knowledge graph context below to answer the question.

Knowledge Graph Context:
{graph_context}

Question: {question}

Options:
{options_text}

Reply with only the letter A, B, C, or D. Nothing else."""

    predicted = "X"
    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5, temperature=0
            )
            text = response.choices[0].message.content.strip()
            for char in text:
                if char in ['A', 'B', 'C', 'D']:
                    predicted = char
                    break
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(10)

    if predicted == "X":
        return "X", 0

    # Step 3: Confidence check
    time.sleep(2)
    confidence = get_answer_confidence(question, options, predicted, graph_context)
    initial_confidence = confidence

    # Step 4: Self-correct if low confidence
    if confidence < confidence_threshold:
        print(f"    Low confidence ({confidence}/10) - triggering self-correction...")
        expanded_context = get_graph_context_semantic(question, top_k=10)

        correction_prompt = f"""You are a medical expert reviewing your previous answer.

Your previous answer was {predicted} but you had low confidence.

Expanded Knowledge Graph Context (including latest PubMed evidence):
{expanded_context}

Question: {question}

Options:
{options_text}

Review carefully and provide your best answer.
Reply with only the letter A, B, C, or D. Nothing else."""

        for attempt in range(retries):
            try:
                response = client_groq.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                        {"role": "user", "content": correction_prompt}
                    ],
                    max_tokens=5, temperature=0
                )
                text = response.choices[0].message.content.strip()
                for char in text:
                    if char in ['A', 'B', 'C', 'D']:
                        predicted = char
                        break
                break
            except Exception as e:
                print(f"Correction attempt {attempt+1} failed: {e}")
                time.sleep(10)

    return predicted, initial_confidence


# Run evaluation
test_sample = dataset['test'].select(range(20))
agentic_pubmed_results = []

for i, item in enumerate(test_sample):
    print(f"\n{i+1}/20 processing...")
    predicted, confidence = ask_agentic_graphrag_pubmed(item['question'], item['options'])
    correct = item['answer_idx']
    agentic_pubmed_results.append({
        'predicted': predicted,
        'correct': correct,
        'is_correct': predicted == correct,
        'confidence': confidence,
    })
    print(f"{i+1}/20 - Predicted: {predicted} | Correct: {correct} | {'✓' if predicted == correct else '✗'}")
    time.sleep(4)

df_agentic_pubmed = pd.DataFrame(agentic_pubmed_results)

print("\n=== FINAL COMPARISON ===")
print(f"Baseline LLM          : 70.0%")
print(f"RAG                   : 65.0%")
print(f"GraphRAG              : 70.0%")
print(f"Agentic GraphRAG      : 75.0%")
print(f"Agentic GraphRAG+PubMed: {df_agentic_pubmed['is_correct'].mean()*100:.1f}%")


1/20 processing...
    Fetching PubMed abstracts...
    PubMed query: Carpal tunnel syndrome, flexor tendon injury, surgical complications, medical ethics
    Abstracts fetched: 0
    Low confidence (1/10) - triggering self-correction...
1/20 - Predicted: A | Correct: B | ✗

2/20 processing...
    Fetching PubMed abstracts...
    PubMed query: transitional cell carcinoma, sensorineural hearing loss, neoadjuvant chemotherapy, cisplatin
    Abstracts fetched: 0
    Low confidence (1/10) - triggering self-correction...
2/20 - Predicted: D | Correct: D | ✓

3/20 processing...
    Fetching PubMed abstracts...
    PubMed query: unstable angina pectoris, contrast-induced nephropathy, type 2 diabetes mellitus, acute kidney injury
    Abstracts fetched: 0
3/20 - Predicted: B | Correct: B | ✓

4/20 processing...
    Fetching PubMed abstracts...
    PubMed query: Disseminated intravascular coagulation, sepsis, pelvic inflammatory disease, thrombocytopenia
    Abstracts fetched: 0
    Low confide

In [27]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import json
import os
import time
from openai import OpenAI

# ============ Set your new key here — only place you need to touch ============
client_groq = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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"
)
# ================================================================================


def fetch_pubmed_abstracts(query, max_results=3):
    """Fetch abstracts from PubMed using NCBI Entrez API (no key needed)"""
    try:
        search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={urllib.parse.quote(query)}&retmax={max_results}&retmode=xml"
        with urllib.request.urlopen(search_url, timeout=10) as response:
            search_xml = response.read()

        root = ET.fromstring(search_xml)
        pmids = [id_elem.text for id_elem in root.findall('.//Id')]

        if not pmids:
            return []

        ids_str = ",".join(pmids)
        fetch_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id={ids_str}&rettype=abstract&retmode=xml"
        with urllib.request.urlopen(fetch_url, timeout=10) as response:
            fetch_xml = response.read()

        root = ET.fromstring(fetch_xml)
        abstracts = []
        for article in root.findall('.//PubmedArticle'):
            abstract_text = ""
            for text in article.findall('.//AbstractText'):
                if text.text:
                    abstract_text += text.text + " "
            if abstract_text.strip():
                abstracts.append(abstract_text.strip()[:500])

        return abstracts
    except Exception as e:
        print(f"    PubMed fetch failed: {e}")
        return []


def extract_pubmed_keywords(question, retries=2):
    """Extract 2-4 clinically searchable keywords from a vignette (MeSH-style, not prose)."""
    prompt = f"""Extract 2 to 4 key medical search terms from this clinical vignette,
suitable for a PubMed search (e.g., disease names, drug names, key symptoms).
Return ONLY a comma-separated list, nothing else.

Vignette: {question[:600]}

Keywords:"""
    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=40,
                temperature=0
            )
            keywords = response.choices[0].message.content.strip().replace("\n", " ")
            if keywords:
                return keywords
        except Exception as e:
            print(f"    Keyword extraction attempt {attempt+1} failed: {e}")
            time.sleep(5)
    return question[:100]


def update_graph_from_pubmed(question, G, node_list, node_embeddings):
    """Fetch PubMed abstracts and dynamically add entities to graph"""
    search_query = extract_pubmed_keywords(question)
    print(f"    PubMed query: {search_query}")
    abstracts = fetch_pubmed_abstracts(search_query)
    print(f"    Abstracts fetched: {len(abstracts)}")

    new_nodes = 0
    new_edges = 0

    for abstract in abstracts:
        extracted = extract_entities_and_relationships(abstract)
        for ent in extracted.get("entities", []):
            if ent["name"] not in G:
                G.add_node(ent["name"], type=ent["type"])
                node_list.append(ent["name"])
                new_nodes += 1
        for rel in extracted.get("relationships", []):
            if rel["source"] in G and rel["target"] in G:
                G.add_edge(rel["source"], rel["target"], relation=rel["relation"])
                new_edges += 1

    if new_nodes > 0:
        new_embeddings = embedder.encode(node_list)
        return new_embeddings, new_nodes, new_edges

    return node_embeddings, new_nodes, new_edges


def ask_agentic_graphrag_pubmed(question, options, retries=3, confidence_threshold=6):
    global node_list, node_embeddings

    print(f"    Fetching PubMed abstracts...")
    node_embeddings, new_nodes, new_edges = update_graph_from_pubmed(
        question, G, node_list, node_embeddings
    )
    if new_nodes > 0:
        print(f"    Graph updated: +{new_nodes} entities, +{new_edges} relationships")

    graph_context = get_graph_context_semantic(question)
    options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are a medical expert. Use the knowledge graph context below to answer the question.

Knowledge Graph Context:
{graph_context}

Question: {question}

Options:
{options_text}

Reply with only the letter A, B, C, or D. Nothing else."""

    predicted = "X"
    for attempt in range(retries):
        try:
            response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=5, temperature=0
            )
            text = response.choices[0].message.content.strip()
            for char in text:
                if char in ['A', 'B', 'C', 'D']:
                    predicted = char
                    break
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(10)

    if predicted == "X":
        return "X", 0

    time.sleep(2)
    confidence = get_answer_confidence(question, options, predicted, graph_context)
    initial_confidence = confidence

    if confidence < confidence_threshold:
        print(f"    Low confidence ({confidence}/10) - triggering self-correction...")
        expanded_context = get_graph_context_semantic(question, top_k=10)

        correction_prompt = f"""You are a medical expert reviewing your previous answer.

Your previous answer was {predicted} but you had low confidence.

Expanded Knowledge Graph Context (including latest PubMed evidence):
{expanded_context}

Question: {question}

Options:
{options_text}

Review carefully and provide your best answer.
Reply with only the letter A, B, C, or D. Nothing else."""

        for attempt in range(retries):
            try:
                response = client_groq.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": "You are a medical expert. Always respond with only a single letter: A, B, C, or D."},
                        {"role": "user", "content": correction_prompt}
                    ],
                    max_tokens=5, temperature=0
                )
                text = response.choices[0].message.content.strip()
                for char in text:
                    if char in ['A', 'B', 'C', 'D']:
                        predicted = char
                        break
                break
            except Exception as e:
                print(f"Correction attempt {attempt+1} failed: {e}")
                time.sleep(10)

    return predicted, initial_confidence


# ============ Run evaluation (with checkpointing built in) ============

test_sample = dataset['test'].select(range(20))

if os.path.exists('/content/pubmed_partial_results.json'):
    with open('/content/pubmed_partial_results.json', 'r') as f:
        agentic_pubmed_results = json.load(f)
    print(f"Resuming: {len(agentic_pubmed_results)} results already saved")
else:
    agentic_pubmed_results = []

start_index = len(agentic_pubmed_results)

for i in range(start_index, 20):
    item = test_sample[i]
    print(f"\n{i+1}/20 processing...")
    predicted, confidence = ask_agentic_graphrag_pubmed(item['question'], item['options'])
    correct = item['answer_idx']
    agentic_pubmed_results.append({
        'predicted': predicted,
        'correct': correct,
        'is_correct': predicted == correct,
        'confidence': confidence,
    })
    print(f"{i+1}/20 - Predicted: {predicted} | Correct: {correct} | {'✓' if predicted == correct else '✗'}")

    with open('/content/pubmed_partial_results.json', 'w') as f:
        json.dump(agentic_pubmed_results, f)

    time.sleep(4)

import pandas as pd
df_agentic_pubmed = pd.DataFrame(agentic_pubmed_results)

print("\n=== FINAL COMPARISON ===")
print(f"Baseline LLM           : 70.0%")
print(f"RAG                    : 65.0%")
print(f"GraphRAG               : 70.0%")
print(f"Agentic GraphRAG       : 75.0%")
print(f"Agentic GraphRAG+PubMed: {df_agentic_pubmed['is_correct'].mean()*100:.1f}%")


1/20 processing...
    Fetching PubMed abstracts...
    PubMed query: Carpal tunnel syndrome, flexor tendon injury, surgical complications, medical ethics
    Abstracts fetched: 0
    Low confidence (1/10) - triggering self-correction...
1/20 - Predicted: A | Correct: B | ✗

2/20 processing...
    Fetching PubMed abstracts...
    PubMed query: transitional cell carcinoma, sensorineural hearing loss, neoadjuvant chemotherapy, cisplatin
    Abstracts fetched: 0
2/20 - Predicted: D | Correct: D | ✓

3/20 processing...
    Fetching PubMed abstracts...
    PubMed query: unstable angina pectoris, contrast-induced nephropathy, type 2 diabetes mellitus, clopidogrel
    Abstracts fetched: 0
3/20 - Predicted: B | Correct: B | ✓

4/20 processing...
    Fetching PubMed abstracts...
    PubMed query: Disseminated intravascular coagulation, sepsis, pelvic inflammatory disease, thrombocytopenia
    Abstracts fetched: 0
    Low confidence (1/10) - triggering self-correction...
4/20 - Predicted: B | C

# New Section